# Lesson 2.9 — 使用 motion planner 的 expert demonstrations

到这里为止的一切使用的都是 **random** actions。本 notebook 产出项目此前从未有过的东西：真正**成功**的 trajectories，因而可以作为 imitation learning 的 expert demonstrations。

该 planner 是 ManiSkill 自带的 sampling-based motion planner，也就是其内置 demonstration 生成所使用的同一组件。recipe 取自随包发布的参考实现：

```text
mani_skill/examples/motionplanning/panda/solutions/pick_cube.py
```

## 塑造本 notebook 的两个结果

**1. planner 需要降级 NumPy。** `mplib` 0.1.1 —— ManiSkill 3.0.1 通过 `Requires-Dist: mplib==0.1.1` 固定了该版本 —— 是针对 NumPy 1.x 的 C API 编译的。在 NumPy 2.x 下其保存的函数指针无效，因此构造 planner 时会跳转到地址 `0x0`，进程以 `SIGSEGV` 结束：

```text
Fatal Python error: Segmentation fault
  File ".../mplib/planner.py", line 65 in __init__
  File ".../base_motionplanner/motionplanner.py", line 59 in setup_planner
  File ".../panda/motionplanner.py", line 25 in __init__
```

Kernel 日志：`segfault at 0 ip 0000000000000000`。修复方式是 `numpy<2`，**不是** GPU 或 backend 设置：同一个脚本在 NumPy 2.2.6 下 segfault，在 NumPy 1.26.4 下成功，而两种情况 CUDA 都不可用。

**2. control mode 与 gripper 步骤是关键的承载环节。** planner 的 `close_gripper()` / `open_gripper()` 发出 `[qpos(7), gripper]`，因此环境必须运行 `pd_joint_pos`。而且官方 recipe 在把 cube 搬运到 goal 之后**不会打开 gripper** —— 加上 `open_gripper()` 以及 settling 步骤会把 cube 丢在 goal 旁边，episode 以 `is_obj_placed: False` 失败。

本 notebook 记录的是实际可行的做法。

## 2.9.1 — 前言

In [1]:
import gymnasium as gym
import mani_skill.envs
import numpy as np
import torch

torch.set_printoptions(precision=4, sci_mode=False)


def make_env(obs_mode="state", control_mode="pd_joint_pos", seed=0):
    env = gym.make(
        "PickCube-v1",
        obs_mode=obs_mode,
        control_mode=control_mode,
        num_envs=1,
    )
    env.reset(seed=seed)
    return env

## 环境隔离

本课使用两个环境：

| 环境 | 用途 |
|---|---|
| embodied | 通用学习、VLA 实验 |
| embodied310 | ManiSkill expert 生成 |

expert 生成环境使用：
- Python 3.10
- NumPy 1.26.4
- mplib 0.1.1

因为 mplib 0.1.1 与 NumPy 2.x 不兼容。

## 2.9.2 — 为什么 planner 无法在本 notebook 的环境中运行

项目有两个 Python 环境，它们恰好在本 notebook 所需的这个库上存在差异：

| 环境 | NumPy | mplib planner |
|---|---|---|
| `embodied`（notebook 默认） | `2.2.6` | **segfault** |
| `embodied310`（planning） | `1.26.4` | 正常工作 |

因此本 notebook **不构造 planner**。这样做会直接杀死 kernel —— segmentation fault 无法被 `try` / `except` 捕获，也没有错误可记录，只有一个已经死掉的进程。

planner 相关工作被委托给 `scripts/generate_expert_demo.py`，用 `embodied310` 运行。在这里可以安全检查的是 planner 所依赖的 **action contract**，而这正是 control mode 不能是 `pd_joint_delta_pos` 的真正原因。

In [2]:
import gymnasium as gym
import mani_skill.envs

# The planner's close_gripper() / open_gripper() step the env with
# action = [qpos(7), gripper]  -- an (8,) ABSOLUTE joint position vector.
# Compare the action space of the two candidate control modes.
for mode in ("pd_joint_pos", "pd_joint_delta_pos"):
    e = gym.make("PickCube-v1", obs_mode="state", control_mode=mode, num_envs=1)
    space = e.action_space
    normalized = bool(np.allclose(space.low, -1.0) and np.allclose(space.high, 1.0))
    print(f"{mode:<20} dim={space.shape[0]}  normalized={normalized}")
    if not normalized:
        print(f"{'':<20} low  = {np.round(space.low, 4)}")
        print(f"{'':<20} high = {np.round(space.high, 4)}")
    e.close()

print()
print("pd_joint_pos       : absolute joint targets inside the joint limits.")
print("                     [qpos(7), gripper] fits directly -- this is what the planner emits.")
print(
    "pd_joint_delta_pos : normalized deltas + absolute gripper. The arm phase of "
    "follow_path still emits 8-d, so it passes the shape check"
)
print("                     and is silently misread as deltas; the gripper helpers emit 15-d")
print("                     and the controller raises:")
print("                     'Received action of shape torch.Size([15]) but expected shape (1, 8)'.")


pd_joint_pos         dim=8  normalized=False
                     low  = [-2.8973 -1.7628 -2.8973 -3.0718 -2.8973 -0.0175 -2.8973 -1.    ]
                     high = [ 2.8973  1.7628  2.8973 -0.0698  2.8973  3.7525  2.8973  1.    ]
pd_joint_delta_pos   dim=8  normalized=True

pd_joint_pos       : absolute joint targets inside the joint limits.
                     [qpos(7), gripper] fits directly -- this is what the planner emits.
pd_joint_delta_pos : normalized deltas + absolute gripper. The arm phase of follow_path still emits 8-d, so it passes the shape check
                     and is silently misread as deltas; the gripper helpers emit 15-d
                     and the controller raises:
                     'Received action of shape torch.Size([15]) but expected shape (1, 8)'.


In [3]:
# ==================================================
# Demonstrate action semantic mismatch
#
# Motion planner generates:
#     absolute joint positions
#
#     [q1, q2, ..., q7, gripper]
#
# However:
#
# pd_joint_delta_pos expects:
#     joint position deltas
#
# They may have the same dimension,
# but represent different meanings.
# ==================================================


env_delta = gym.make(
    "PickCube-v1",
    obs_mode="state",
    control_mode="pd_joint_delta_pos",
    num_envs=1
)


obs, info = env_delta.reset(seed=0)


print("Delta controller action space:")
print(env_delta.action_space)


# --------------------------------------------------
# A valid delta action
#
# Meaning:
#     move joints slightly from current position
# --------------------------------------------------

delta_action = np.zeros(
    env_delta.action_space.shape,
    dtype=np.float32
)


obs, reward, terminated, truncated, info = env_delta.step(
    delta_action
)


print("-----------------------------")
print("Delta action accepted")
print("Reward:", reward)


env_delta.close()

Delta controller action space:
Box(-1.0, 1.0, (8,), float32)
-----------------------------
Delta action accepted
Reward: tensor([0.0653])


## 重要：action 语义

`pd_joint_delta_pos` 与 `pd_joint_pos` 可能有相同的 action 维度，但它们代表不同的 control 命令。

---

### `pd_joint_pos`

**Action：**

```text
[q1, q2, ..., q7, gripper]
```

**含义：**

把机器人移动到这些目标 joint positions。

该 action 表示 **absolute 的目标 joint 配置**。

---

### `pd_joint_delta_pos`

**Action：**

```text
[Δq1, Δq2, ..., Δq7, gripper]
```

**含义：**

按这些增量改变当前的 joint positions。

该 action 表示相对于当前 state 的 **relative joint 运动**。

---

### 专家演示（expert demonstration）

`mplib` motion planner 生成 absolute 的 joint trajectories：

```text
current robot state
        |
        v
   mplib planner
        |
        v
[q1, q2, ..., q7, gripper]
```

因此，expert demonstrations 使用的是：

```text
pd_joint_pos
```

而不是：

```text
pd_joint_delta_pos
```

### planner 如何执行它的 actions

`move_to_pose_with_screw` 规划 joint 空间的运动，并通过 `env.step` 驱动每一个 waypoint —— 它不会绕开环境直接设置 joint positions。`TwoFingerGripperMotionPlanningSolver.follow_path` 构造 `[qpos(7), gripper]`，即 8 维的 **absolute** joint target，并对每个规划出的 waypoint 调用 `self.env.step(action)`。`open_gripper()` 与 `close_gripper()` 也以同样的方式 step 环境。

这对数据采集很重要：planner 产出的是 **trajectories**，而把 trajectory 变成训练数据需要环境实际执行过的 actions。collector 脚本同时包装了 `env.step` 与 `env.reset`，因此每个 transition 都在发生时被记录，而不是事后重建 —— 见 2.9.6 节。

这也解释了为什么 control mode 是关键承载环节，因为这两个方法针对不同的 mode 做分支判断：

| 方法 | 8-d 分支 | 15-d 分支 | 15-d 布局 |
|---|---|---|---|
| `follow_path` | 除 `pd_joint_pos_vel` 之外的所有 mode | `pd_joint_pos_vel` | `[qpos(7), qvel(7), gripper]`（真实 velocities） |
| `open_gripper` / `close_gripper` | 仅 `pd_joint_pos` | 其他所有 mode | `[qpos(7), qpos*0(7), gripper]`（**zeros**） |

在 `pd_joint_delta_pos` 下，机械臂阶段仍然发出 8 维 actions：shape 检查通过，但 absolute 的 joint targets 被当作 normalized deltas 读取，因此运动在无声中出错。该失败只在第一次调用 gripper 时显现，此时 15 维向量被拒绝。那个 15 维向量也不是 `pd_joint_pos_vel` 的 action —— 它的中间块是 zeros，而不是 `qvel`。

## 2.9.3 — 抓取几何

一个 grasp pose 需要三样东西，而它们都不是魔法数字：

- **approaching**：gripper 接近的方向。垂直向下，因为 cube 放在桌面上。
- **closing**：手指闭合所沿的轴，由物体的 oriented bounding box 导出。正是这一点让抓取对处于任意 yaw 的 cube 仍然有效。
- **center**：抓取的位置。

`target_closing` 取自 TCP 自身的旋转：复用机械臂当前的 finger 轴，而不是另造一个。`build_grasp_pose` 返回的是 **SAPIEN** `Pose`，而不是 ManiSkill `Pose` —— 因此它直接暴露 `.p` 与 `.q`，而不是 `.raw_pose`。

In [4]:
import sapien

from mani_skill.examples.motionplanning.base_motionplanner.utils import (
    compute_grasp_info_by_obb,
    get_actor_obb,
)

FINGER_LENGTH = 0.025

env = make_env(obs_mode="state", control_mode="pd_joint_pos", seed=0)
unwrapped = env.unwrapped

obb = get_actor_obb(unwrapped.cube)

approaching = np.array([0, 0, -1])
# y axis of the TCP frame = the finger closing axis
target_closing = unwrapped.agent.tcp.pose.to_transformation_matrix()[0, :3, 1].cpu().numpy()

grasp_info = compute_grasp_info_by_obb(
    obb,
    approaching=approaching,
    target_closing=target_closing,
    depth=FINGER_LENGTH,
)
closing = grasp_info["closing"]
center = grasp_info["center"]
grasp_pose = unwrapped.agent.build_grasp_pose(approaching, closing, unwrapped.cube.pose.sp.p)

print("cube position :", unwrapped.cube.pose.sp.p)
print("goal position :", unwrapped.goal_site.pose.sp.p)
print("approaching   :", approaching)
print("closing axis  :", np.round(closing, 4))
print("grasp center  :", np.round(center, 4))
print("grasp pose type:", type(grasp_pose).__name__, "(SAPIEN Pose)")
print("grasp pose p  :", np.round(grasp_pose.p, 4))
print("grasp pose q  :", np.round(grasp_pose.q, 4))

# Reach pose: grasp pose offset 5 cm upward, so the gripper descends from above
reach_pose = grasp_pose * sapien.Pose([0, 0, -0.05])
print("reach  pose p :", np.round(reach_pose.p, 4))

# Goal pose reuses the grasp orientation so the cube stays upright on the way over
goal_pose = sapien.Pose(unwrapped.goal_site.pose.sp.p, grasp_pose.q)
print("goal   pose p :", np.round(goal_pose.p, 4))

cube position : [-0.00074868  0.05364437  0.02      ]
goal position : [ 0.02681573 -0.00198132  0.28893346]
approaching   : [ 0  0 -1]
closing axis  : [ 0.353  -0.9356  0.    ]
grasp center  : [-0.0007  0.0536  0.02  ]
grasp pose type: Pose (SAPIEN Pose)
grasp pose p  : [-0.0007  0.0536  0.02  ]
grasp pose q  : [0.     0.9838 0.1794 0.    ]
reach  pose p : [-0.0007  0.0536  0.07  ]
goal   pose p : [ 0.0268 -0.002   0.2889]


## 2.9.4 — 执行计划

经验证的序列是 **四步，不能再多**：

```python
planner.move_to_pose_with_screw(reach_pose)   # approach from above
planner.move_to_pose_with_screw(grasp_pose)   # descend
planner.close_gripper()                       # grasp
planner.move_to_pose_with_screw(goal_pose)    # carry to the goal
```

有两个负面结果值得说明，因为它们看起来都合理，却都失败了：

- **搬运之后调用 `open_gripper()`** —— cube 在仍然紧挨着 goal 时被释放，未通过放置距离检查，`is_obj_placed` 返回 `False`。
- **额外的 zero-action settling 步骤** —— 它们无法修复放置；反而让 cube 漂移得更远。

注意 `close_gripper()` 与 `open_gripper()` 内部会用形如 `[qpos(7), gripper]` 的 `(8,)` action 调用 `env.step`。这就是环境必须是 `pd_joint_pos` 的原因：gripper helpers 只在该 mode 下走 8 维分支，而在 `pd_joint_delta_pos` 下它们发出 15 维向量，controller 会以 `AssertionError: Received action of shape torch.Size([15]) but expected shape (1, 8)` 拒绝它。`follow_path` 的表现不同 —— 它在 delta mode 下仍保持 8 维，因此机械臂运动被无声地误读，直到 gripper 调用最终失败（2.9.2 节）。

In [5]:
import subprocess
import sys
from pathlib import Path

# The planner must run in embodied310 (numpy<2), so invoke the collector as a
# subprocess rather than constructing the planner in this kernel.
PLANNING_PYTHON = Path.home() / "miniforge3" / "envs" / "embodied310" / "bin" / "python"
from pathlib import Path

# Notebooks may be started with either the repository root or notebooks/ as the working
# directory. Walk up until the project root is found, so dataset and cache paths never
# resolve to notebooks/datasets or notebooks/.cache by accident.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / ".git").exists()),
    _cwd.parent if _cwd.name == "notebooks" else _cwd,
)
COLLECTOR = PROJECT_ROOT / "scripts" / "generate_expert_demo.py"

print("planning interpreter:", PLANNING_PYTHON, "exists:", PLANNING_PYTHON.exists())

if PLANNING_PYTHON.exists():
    result = subprocess.run(
        [str(PLANNING_PYTHON), str(COLLECTOR), "--seeds", "0", "1", "2", "3", "4", "--overwrite"],
        capture_output=True, text=True, cwd=str(PROJECT_ROOT),
    )
    print("exit code:", result.returncode)
    print(result.stdout.strip()[-1200:])
    if result.returncode != 0:
        print("STDERR:", result.stderr.strip()[-600:])
else:
    print("embodied310 not found; run the collector manually:")
    print("  python scripts/generate_expert_demo.py --seeds 0 1 2 3 4 --overwrite")

planning interpreter: /home/bowenyuan/miniforge3/envs/embodied310/bin/python exists: True
exit code: 0
environment : PickCube-v1
control mode: pd_joint_pos
seeds       : [0, 1, 2, 3, 4]
seed 0: actions=74 observations=75 success=True placed=True static=True return=24.9710
seed 1: actions=74 observations=75 success=True placed=True static=True return=23.1076
seed 2: actions=50 observations=51 success=True placed=True static=True return=14.7990
seed 3: actions=86 observations=87 success=True placed=True static=True return=27.8694
seed 4: actions=76 observations=77 success=True placed=True static=True return=25.6839

successful episodes: 5/5
written: /home/bowenyuan/Projects/embodied-ai-learning/datasets/pickcube/expert_episodes.h5

NOTE: this is expert data in pd_joint_pos semantics. The project's existing fixture is pd_joint_delta_pos; see notes/progress.md for how the two relate.


## 2.9.5 — 成功的含义

成功是 **task evaluation** 的属性，而不是 planner 无错误返回的属性：

```python
info = env.unwrapped.evaluate()
# {"success", "is_obj_placed", "is_robot_static", "is_grasped"}
```

| 字段 | 含义 |
|---|---|
| `is_obj_placed` | cube 位于 goal 的 `goal_thresh`（0.025 m）范围内 |
| `is_robot_static` | 机械臂已稳定 |
| `success` | `is_obj_placed AND is_robot_static` |

**抓取并不等于成功。** 机械臂在半空中握住 cube 并没有完成任务。这就是为什么在本项目任何地方，“trajectory 看起来合理”都不是验收标准。

In [6]:
print("goal threshold:", unwrapped.goal_thresh, "m")
print("cube half size:", unwrapped.cube_half_size)
print()
print("evaluate() keys:", list(unwrapped.evaluate().keys()))
print("success requires BOTH is_obj_placed AND is_robot_static.")

goal threshold: 0.025 m
cube half size: 0.02

evaluate() keys: ['success', 'is_obj_placed', 'is_robot_static', 'is_grasped']
success requires BOTH is_obj_placed AND is_robot_static.


## 2.9.6 — 采集到的 expert episodes

`scripts/generate_expert_demo.py` 把这个 recipe 封装成一个 collector。它同时包装了 `env.step` 与 `env.reset`，因此每个 transition 都在执行时被记录 —— 没有任何内容是事后重建的 —— 并且在记录之前会断言 control mode 与 action 维度。

记录的 schema，其中执行了 `T` 个 actions：

```text
observations[T + 1] = o_0 .. o_T      # o_t is the state before a_t
actions[T]          = a_0 .. a_{T-1}  # executed by env.step
rewards[T]          = r_0 .. r_{T-1}
```

`o_0` 是 reset observation，训练对是 `observations[:-1]` 与 `actions`，而 `observations[1:]` 是与之匹配的 next states。这个 collector 的第一个版本只包装 `env.step`，存储每一步*返回的* observation，因此保存了 `(o_{t+1}, a_t)` 而丢掉了 `o_0`；该缺陷已在此修复，文件也已重新生成。

seeds 0–4 的记录结果：

| Seed | Actions `T` | Observations `T+1` | 成功 | is_obj_placed | is_robot_static |
|---:|---:|---:|:--:|:--:|:--:|
| 0 | 74 | 75 | True | True | True |
| 1 | 74 | 75 | True | True | True |
| 2 | 50 | 51 | True | True | True |
| 3 | 86 | 87 | True | True | True |
| 4 | 76 | 77 | True | True | True |

**5/5 的 episodes 成功**，并且只写入成功的 episodes：失败的 seed 会报告到 stderr 并被排除，被排除的 seeds 记录在 root attribute `failed_seeds` 中。

比较 action 的平滑程度：

| Dataset | mean `|Δa|` |
|---|---|
| random rollout fixture | `0.67` |
| expert planner episodes | `0.0078` |

这大约是 **100× 的差异**。expert actions 是小的、彼此相关的步进；random actions 则几乎是独立抽样。仅这一个统计量就把两种情形区分开来，这也是为什么 random trajectory 无论训练多久都无法训练出 policy。

collector 还会把 contract 写入 HDF5 attributes：`control_mode`、`data_quality="expert_planner"`、`control_freq_hz=20`、`timestamp_source="derived_not_measured"`、`transition_schema`，以及各通道的 action 语义。

In [7]:
import h5py
from pathlib import Path

# Notebooks may be started with either the repository root or notebooks/ as the working
# directory. Walk up until the project root is found, so dataset and cache paths never
# resolve to notebooks/datasets or notebooks/.cache by accident.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / ".git").exists()),
    _cwd.parent if _cwd.name == "notebooks" else _cwd,
)
EXPERT_H5 = PROJECT_ROOT / "datasets" / "pickcube" / "expert_episodes.h5"

if not EXPERT_H5.exists():
    print(f"{EXPERT_H5} not found.")
    print("Generate it with: python scripts/generate_expert_demo.py --seeds 0 1 2 3 4 --overwrite")
else:
    with h5py.File(EXPERT_H5, "r") as handle:
        print("root attributes:")
        for key in ("env_id", "control_mode", "data_quality", "control_freq_hz",
                    "timestamp_source", "transition_schema_version", "failed_seeds"):
            print(f"  {key:<26} {handle.attrs[key]}")
        print()
        print(f"{'episode':<16} {'acts':>4} {'obs':>4} {'success':>8} {'|da| mean':>10} {'return':>8}")
        print("-" * 56)
        for name in handle:
            group = handle[name]
            observations = group["observations"][:]
            actions = group["actions"][:]
            rewards = group["rewards"][:]
            # The schema invariant: T actions, T rewards, T + 1 observations.
            assert observations.shape[0] == actions.shape[0] + 1 == rewards.shape[0] + 1, name
            smoothness = np.abs(np.diff(actions, axis=0)).mean()
            print(f"{name:<16} {actions.shape[0]:>4} {observations.shape[0]:>4} "
                  f"{str(bool(group.attrs['success'])):>8} "
                  f"{smoothness:>10.4f} {rewards.sum():>8.3f}")


root attributes:
  env_id                     PickCube-v1
  control_mode               pd_joint_pos
  data_quality               expert_planner
  control_freq_hz            20
  timestamp_source           derived_not_measured
  transition_schema_version  2
  failed_seeds               []

episode          acts  obs  success  |da| mean   return
--------------------------------------------------------
episode_000000     74   75     True     0.0078   24.971
episode_000001     74   75     True     0.0074   23.108
episode_000002     50   51     True     0.0080   14.799
episode_000003     86   87     True     0.0077   27.869
episode_000004     76   77     True     0.0077   25.684


## 2.9.7 — replay 验证

expert dataset 只有在 replay 能复现所记录的结果时才会被接受。本机上运行了两项独立检查：

| 检查 | 结果 |
|---|---|
| Action replay：`reset(seed)`，喂入存储的 `actions`，逐步比较 | observations 与 rewards **完全**复现（`max error 0.0`）；`success` 一致（5/5） |
| Planner 重跑：在全新进程中重新生成 seeds 0–4 | `actions`、`rewards` 与 `observations` 与上一次运行 **bit-identical** |
| 存储的 `o_0` 与 `env.reset(seed)` 的 observation | 精确一致（`max |diff| = 0.0`） |

本 notebook 的早期版本曾报告 planner 重跑只能吻合到 `~1e-2`，并且布尔量 `is_grasped` 在 74 帧中有一帧翻转。这两者在当前都不可复现，可能的原因是本次修订所修复的 schema 缺陷：旧的 collector 存储 `observations[t] = o_{t+1}`，因此把 replay 在索引 `t` 处的 state 与存储的 state 比较时，contact 布尔量恰好错开一帧。物理本身从未可疑；错的是索引。

教训：当 replay “几乎”吻合时，先检查 transition schema，再去归咎 solver。要说明你实际验证的容差 —— 这里是精确相等，因为现在的比较已按索引对齐。

## 2.9.8 — 与现有 fixture 的关系

仓库现在保存着两类 trajectory，它们**不可**互换：

| | `random_episode_standard.h5` | `expert_episodes.h5` |
|---|---|---|
| 来源 | random actions | motion planner |
| Episodes | 1 | 5 |
| Actions `T` | 50 | 74 / 74 / 50 / 86 / 76 |
| Observations | 50（`o_t`，pre-action） | `T + 1`（`o_0 .. o_T`） |
| 成功 | none（`success_any=False`） | 5/5 |
| Control mode | `pd_joint_delta_pos` | `pd_joint_pos` |
| Action dim | 8（7 delta + 1 absolute） | 8（8 absolute） |
| 作用 | pipeline fixture | imitation-learning supervision |

**action 语义不同**，尽管两者都是 8 维、都位于一个 box 中。`pd_joint_delta_pos` 给出的机械臂 **deltas** 在 `[-0.1, 0.1]` rad 范围内，外加一个 absolute 的 gripper target；`pd_joint_pos` 给出 joint limits 内的 **absolute** joint position targets。不做转换就把它们拼接起来，会让 policy 在两种不兼容的 action 语义上训练 —— 这正是 `1.2_action_space_and_control_modes.ipynb` 所警告的失败模式。

transition schemas 也不同：fixture 存储 `T` 个 pre-action observations 外加一个单独的 `next_observations` 数组，而 expert 文件存储 `T + 1` 个 observations，且没有单独的 next-state 数组。在不了解其约定（convention）的情况下读取任何一个，都会引入无声的一帧偏移。

从 absolute targets 反推 deltas 的 converter 很直接（`Δq_t = q_target[t] - qpos[t]`，然后乘以 `0.1` 并 clamp 到 `[-1,1]`），但在把两个 datasets 混用之前，必须先构建并验证它。这里刻意**不**做这件事。

## 小结

1. 一个 expert demonstration 来自 **planner**，而不是 policy。random rollouts 是 pipeline fixtures；planners 产出 supervision。
2. `mplib` 0.1.1 要求 `numpy<2`。在 NumPy 2.x 下它会在地址 `0x0` 处 segfault，因为其 C-API 函数指针无效。这是 ABI 问题，不是 GPU 问题。
3. planner 需要 `pd_joint_pos`；它的 gripper helpers 发出 absolute position actions。
4. 四步即可成功：reach → grasp → close → carry。加入 `open_gripper()` 或额外的 settling 步骤会破坏放置。
5. 成功是 `is_obj_placed AND is_robot_static`。仅靠抓取不是成功。
6. expert episodes 比 random 的平滑约 100×（`|Δa|` 0.0078 vs 0.67）。
7. replay 现在是精确的：action replay 与独立的 planner 重跑都能逐 bit 复现 observations 与 rewards。早先的“一帧 `is_grasped` 翻转”是旧 `(o_{t+1}, a_t)` schema 的索引产物，而不是 solver 噪声。
8. `pd_joint_pos` 的 expert 数据**不能**直接与现有的 `pd_joint_delta_pos` fixture 拼接。先转换并验证。

## 2.9.9 — 校验 expert transition 契约

只有当一个 dataset 为 `T` 个 action 存储 `T + 1` 个 observation 时，它才可被接受：
于是 transition `t` 表示为

\[
(o_t, a_t, r_t, o_{t+1})
\]

shape 检查验证 schema，确定性 replay 验证时间对齐。


In [8]:
from pathlib import Path

import h5py
import numpy as np


# Locate repository root
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / ".git").exists()),
    _cwd.parent if _cwd.name == "notebooks" else _cwd,
)

EXPERT_H5 = (
    PROJECT_ROOT
    / "datasets"
    / "pickcube"
    / "expert_episodes.h5"
)

assert EXPERT_H5.exists(), f"Dataset not found: {EXPERT_H5}"

EXPECTED_OBS_DIM = 42
EXPECTED_ACTION_DIM = 8


with h5py.File(EXPERT_H5, "r") as handle:
    print("Dataset:", EXPERT_H5)
    print("Episodes:", len(handle))
    print()

    for episode_name in sorted(handle.keys()):
        group = handle[episode_name]

        observations = group["observations"][:]
        actions = group["actions"][:]
        rewards = group["rewards"][:]

        T = len(actions)

        # ---------------------------------------
        # Structural contract
        # ---------------------------------------
        assert observations.ndim == 2, (
            f"{episode_name}: observations must be 2-D, "
            f"got {observations.shape}"
        )
        assert actions.ndim == 2, (
            f"{episode_name}: actions must be 2-D, "
            f"got {actions.shape}"
        )
        assert rewards.ndim == 1, (
            f"{episode_name}: rewards must be 1-D, "
            f"got {rewards.shape}"
        )

        assert len(observations) == T + 1, (
            f"{episode_name}: expected T+1 observations, "
            f"got observations={len(observations)}, actions={T}"
        )
        assert len(rewards) == T, (
            f"{episode_name}: rewards/actions mismatch: "
            f"{len(rewards)} vs {T}"
        )

        # ---------------------------------------
        # Feature dimensions
        # ---------------------------------------
        assert observations.shape[1] == EXPECTED_OBS_DIM, (
            f"{episode_name}: expected {EXPECTED_OBS_DIM}-D observation, "
            f"got {observations.shape[1]}"
        )
        assert actions.shape[1] == EXPECTED_ACTION_DIM, (
            f"{episode_name}: expected {EXPECTED_ACTION_DIM}-D action, "
            f"got {actions.shape[1]}"
        )

        # ---------------------------------------
        # Numerical validity
        # ---------------------------------------
        assert np.isfinite(observations).all(), (
            f"{episode_name}: observations contain NaN/Inf"
        )
        assert np.isfinite(actions).all(), (
            f"{episode_name}: actions contain NaN/Inf"
        )
        assert np.isfinite(rewards).all(), (
            f"{episode_name}: rewards contain NaN/Inf"
        )

        # ---------------------------------------
        # Episode validity
        # ---------------------------------------
        success = bool(group.attrs["success"])
        assert success, f"{episode_name}: unsuccessful episode in expert data"

        if "terminated" in group:
            assert len(group["terminated"]) == T

        if "truncated" in group:
            assert len(group["truncated"]) == T

        print(
            f"{episode_name}: PASS | "
            f"obs={observations.shape} "
            f"actions={actions.shape} "
            f"rewards={rewards.shape} "
            f"success={success}"
        )

print()
print("PASS: every episode satisfies the T+1 observation contract.")
print("BC input : observations[:-1]")
print("BC target: actions")
print("Next obs : observations[1:]")

Dataset: /home/bowenyuan/Projects/embodied-ai-learning/datasets/pickcube/expert_episodes.h5
Episodes: 5

episode_000000: PASS | obs=(75, 42) actions=(74, 8) rewards=(74,) success=True
episode_000001: PASS | obs=(75, 42) actions=(74, 8) rewards=(74,) success=True
episode_000002: PASS | obs=(51, 42) actions=(50, 8) rewards=(50,) success=True
episode_000003: PASS | obs=(87, 42) actions=(86, 8) rewards=(86,) success=True
episode_000004: PASS | obs=(77, 42) actions=(76, 8) rewards=(76,) success=True

PASS: every episode satisfies the T+1 observation contract.
BC input : observations[:-1]
BC target: actions
Next obs : observations[1:]


In [9]:
import gymnasium as gym
import mani_skill.envs
import numpy as np
import h5py


def to_numpy_flat(value):
    if hasattr(value, "detach"):
        value = value.detach().cpu().numpy()
    return np.asarray(value, dtype=np.float32).reshape(-1)


EPISODE_NAME = "episode_000000"
OBS_ATOL = 1e-5
REWARD_ATOL = 1e-6


with h5py.File(EXPERT_H5, "r") as handle:
    group = handle[EPISODE_NAME]

    stored_observations = group["observations"][:]
    stored_actions = group["actions"][:]
    stored_rewards = group["rewards"][:]
    seed = int(group.attrs["seed"])

    env_id = handle.attrs["env_id"]
    obs_mode = handle.attrs["obs_mode"]
    control_mode = handle.attrs["control_mode"]


env = gym.make(
    env_id,
    obs_mode=obs_mode,
    control_mode=control_mode,
    num_envs=1,
)

try:
    reset_observation, reset_info = env.reset(seed=seed)
    reset_observation = to_numpy_flat(reset_observation)

    initial_error = np.max(
        np.abs(reset_observation - stored_observations[0])
    )

    max_post_observation_error = 0.0
    max_reward_error = 0.0

    for t, action in enumerate(stored_actions):
        observation, reward, terminated, truncated, info = env.step(action)

        observation = to_numpy_flat(observation)
        reward = float(to_numpy_flat(reward)[0])

        # action[t] must lead from observations[t]
        # to observations[t + 1]
        post_observation_error = np.max(
            np.abs(observation - stored_observations[t + 1])
        )
        reward_error = abs(reward - float(stored_rewards[t]))

        max_post_observation_error = max(
            max_post_observation_error,
            post_observation_error,
        )
        max_reward_error = max(
            max_reward_error,
            reward_error,
        )

    evaluation = env.unwrapped.evaluate()
    replay_success = bool(
        to_numpy_flat(evaluation["success"])[0]
    )

finally:
    env.close()


print("Episode:", EPISODE_NAME)
print("Seed:", seed)
print("Transitions:", len(stored_actions))
print(f"Initial observation error : {initial_error:.8e}")
print(f"Maximum post-step error   : {max_post_observation_error:.8e}")
print(f"Maximum reward error      : {max_reward_error:.8e}")
print("Replay success            :", replay_success)

assert initial_error <= OBS_ATOL, (
    "Stored observations[0] does not match reset observation o0."
)
assert max_post_observation_error <= OBS_ATOL, (
    "Stored observations[t+1] does not match env.step(actions[t])."
)
assert max_reward_error <= REWARD_ATOL, (
    "Stored rewards are not aligned with actions."
)
assert replay_success, "Replayed episode did not succeed."

print()
print("PASS: transition alignment is correct:")
print("observations[t] --actions[t]--> observations[t+1]")

Episode: episode_000000
Seed: 0
Transitions: 74
Initial observation error : 0.00000000e+00
Maximum post-step error   : 0.00000000e+00
Maximum reward error      : 0.00000000e+00
Replay success            : True

PASS: transition alignment is correct:
observations[t] --actions[t]--> observations[t+1]
